In [1]:
from dotenv import load_dotenv
from deepagents.backends import StateBackend
from deepagents import create_deep_agent
from langchain_core.utils.uuid import uuid7
import csv
import io
from langchain.tools import tool


from langchain_openai import ChatOpenAI
import os

load_dotenv()  # loads .env file into environment variables

api_key = os.getenv("ZEN_API_KEY")

backend = StateBackend()

/Users/roypsr/projects/gen_ai_mind/.venv/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1
python-dotenv could not parse statement starting at line 4


In [2]:
import getpass
import os

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = getpass.getpass()
os.environ["LANGSMITH_PROJECT"] = "deep_agent_data_analysis"  # Optional

 ········


In [3]:
# Create sample sales data

@tool
def initialize_workspace():
    """Upload files to the StateBackend."""
    data = [
        ["Date", "Product Name", "Sold Items", "Profit"],
        ["2026-06-01", "Item A", 10, 30],
        ["2026-06-02", "Item B", 5, 20],
        ["2026-06-03", "Item A", 7, 21],
        ["2026-06-04", "Item C", 3, 27],
        ["2026-06-05", "Item B", 8, 32],
    ]
    
    # Convert to CSV bytes
    text_buf = io.StringIO()
    writer = csv.writer(text_buf)
    writer.writerows(data)
    csv_bytes = text_buf.getvalue().encode("utf-8")
    text_buf.close()
    
    # Upload to backend
    backend.upload_files([("/root/data/sales_data.csv", csv_bytes)])

In [4]:
llm = ChatOpenAI(
    model="big-pickle", # Replace with your chosen Zen model id
    openai_api_key=api_key,
    openai_api_base="https://opencode.ai/zen/v1"
)

In [5]:
agent = create_deep_agent(
    model=llm,
    tools=[initialize_workspace],
    backend=backend,
)

In [6]:
thread_id = str(uuid7())
config = {"configurable": {"thread_id": thread_id}}

In [12]:
input_message = {
    "role": "user",
    "content": (
        "Analyze ./data/sales_data.csv in the current dir"
    ),
}
stream = agent.stream_events(
    {"messages": [input_message]},
    config,
    version="v3",
)
for snapshot in stream.values:
    snapshot["messages"][-1].pretty_print()

================================ Human Message =================================

Analyze ./data/sales_data.csv in the current dir
================================== Ai Message ==================================

[{'type': 'tool_call', 'id': 'call_00_dcvV7zSKKVhAv1zBZxEw3192', 'name': 'ls', 'args': {'path': '/'}}, {'type': 'tool_call', 'id': 'call_01_iyT2Ls50x9TE9tWKaY8g8129', 'name': 'glob', 'args': {'pattern': '**/sales_data.csv'}}]
Tool Calls:
  ls (call_00_dcvV7zSKKVhAv1zBZxEw3192)
 Call ID: call_00_dcvV7zSKKVhAv1zBZxEw3192
  Args:
    path: /
  glob (call_01_iyT2Ls50x9TE9tWKaY8g8129)
 Call ID: call_01_iyT2Ls50x9TE9tWKaY8g8129
  Args:
    pattern: **/sales_data.csv
================================= Tool Message =================================
Name: glob

[]
================================== Ai Message ==================================

[{'type': 'tool_call', 'id': 'call_00_GvTogJZvsxXis9HwComV9979', 'name': 'ls', 'args': {'path': './data'}}, {'type': 'tool_call', 'id': 'call_01

In [7]:
input_message = {
    "role": "user",
    "content": (
        "Analyze ./data/sales_data.csv in the current dir"
    ),
}
result = agent.invoke({"messages": [input_message]})

In [8]:
# Print the agent's response
print(result["messages"][-1].content)

The file has 5 transaction records (June 1–5, 2026) with columns `Date, Product Name, Sold Items, Profit`. Here's the analysis:

## Summary
- **Total units sold:** 33
- **Total profit:** $130
- **Average daily sales:** 6.6 units
- **Average daily profit:** $26

## By Product

| Product | Records | Units Sold | Profit | Profit/Unit |
|---------|---------|-----------|--------|-------------|
| Item A  | 2       | 17        | $51    | $3.00       |
| Item B  | 2       | 13        | $52    | $4.00       |
| Item C  | 1       | 3         | $27    | $9.00       |

## Key Insights
- **Highest total profit:** Item B ($52), despite fewer units than Item A — better margin per unit ($4 vs $3).
- **Best margin:** Item C at $9/unit — over 2x Item B and 3x Item A, but lowest volume (only 3 units).
- **Best single day (profit):** 2026-06-05 — Item B, 8 units, $32 profit.
- **Best single day (volume):** 2026-06-01 — Item A, 10 units.
- **No negative or zero-profit transactions** — every line is profita